## Pydantic with gemini LLM
- using Pydantic to structure output from Gemini

In [6]:
from dotenv import load_dotenv
import os 
from google import genai

load_dotenv()

client= genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
response=client.models.generate_content(model = "gemini-2.5-flash", contents= "tell me a programming joke")
print(response.text)

Here's a classic:

A programmer's wife tells him, "Go to the store and buy a loaf of bread. If they have eggs, buy a dozen."

The programmer comes home with 12 loaves of bread.


In [9]:
response=client.models.generate_content(
    model = "gemini-2.5-flash", contents= """
    You are a helpful assistant. I need you to create a JSON object representing a library.
    The library's name should be "Coolu Libraru" and have the fields name and books that
    contains a list of books.
    Each book should have a 'title', 'author', and 'year' field. Make sure the output is a single, valid JSON object. Give me 10 books. 
    Remove ```json and ``` 
"""
    )

response.text

'{\n  "name": "Coolu Libraru",\n  "books": [\n    {\n      "title": "The Silent Forest",\n      "author": "Elara Vance",\n      "year": 2018\n    },\n    {\n      "title": "Whispers of the Cosmos",\n      "author": "Dr. Kaelen Thorne",\n      "year": 2021\n    },\n    {\n      "title": "Chronicles of Aethel",\n      "author": "Lyra Sterling",\n      "year": 2015\n    },\n    {\n      "title": "The Gilded Compass",\n      "author": "Arthur Pendelton",\n      "year": 2019\n    },\n    {\n      "title": "Echoes of Tomorrow",\n      "author": "Seraphina Moon",\n      "year": 2023\n    },\n    {\n      "title": "Beneath the Obsidian Sky",\n      "author": "Roric Stone",\n      "year": 2017\n    },\n    {\n      "title": "The Alchemist\'s Secret",\n      "author": "Isabelle Dubois",\n      "year": 2010\n    },\n    {\n      "title": "Gardens of the Sunken City",\n      "author": "Leo Maritime",\n      "year": 2022\n    },\n    {\n      "title": "A Flicker in the Dark",\n      "author": "Nadi

In [10]:
print(response.text)

{
  "name": "Coolu Libraru",
  "books": [
    {
      "title": "The Silent Forest",
      "author": "Elara Vance",
      "year": 2018
    },
    {
      "title": "Whispers of the Cosmos",
      "author": "Dr. Kaelen Thorne",
      "year": 2021
    },
    {
      "title": "Chronicles of Aethel",
      "author": "Lyra Sterling",
      "year": 2015
    },
    {
      "title": "The Gilded Compass",
      "author": "Arthur Pendelton",
      "year": 2019
    },
    {
      "title": "Echoes of Tomorrow",
      "author": "Seraphina Moon",
      "year": 2023
    },
    {
      "title": "Beneath the Obsidian Sky",
      "author": "Roric Stone",
      "year": 2017
    },
    {
      "title": "The Alchemist's Secret",
      "author": "Isabelle Dubois",
      "year": 2010
    },
    {
      "title": "Gardens of the Sunken City",
      "author": "Leo Maritime",
      "year": 2022
    },
    {
      "title": "A Flicker in the Dark",
      "author": "Nadia Volkov",
      "year": 2016
    },
    {
    

### Use Pydantic to validate the simulated data

In [13]:
from pydantic import BaseModel, Field
from datetime import datetime

class Book(BaseModel):
    title: str
    author: str
    year: int = Field(gt = 1000, lt = datetime.now().year) # gt=greater than, lt= less than

class Library(BaseModel):
    name: str
    books: list[Book]

library = Library.model_validate_json(response.text)
library


Library(name='Coolu Libraru', books=[Book(title='The Silent Forest', author='Elara Vance', year=2018), Book(title='Whispers of the Cosmos', author='Dr. Kaelen Thorne', year=2021), Book(title='Chronicles of Aethel', author='Lyra Sterling', year=2015), Book(title='The Gilded Compass', author='Arthur Pendelton', year=2019), Book(title='Echoes of Tomorrow', author='Seraphina Moon', year=2023), Book(title='Beneath the Obsidian Sky', author='Roric Stone', year=2017), Book(title="The Alchemist's Secret", author='Isabelle Dubois', year=2010), Book(title='Gardens of the Sunken City', author='Leo Maritime', year=2022), Book(title='A Flicker in the Dark', author='Nadia Volkov', year=2016), Book(title="The Cartographer's Dream", author='Elias Croft', year=2020)])

In [14]:
library.name

'Coolu Libraru'

In [15]:
library.books

[Book(title='The Silent Forest', author='Elara Vance', year=2018),
 Book(title='Whispers of the Cosmos', author='Dr. Kaelen Thorne', year=2021),
 Book(title='Chronicles of Aethel', author='Lyra Sterling', year=2015),
 Book(title='The Gilded Compass', author='Arthur Pendelton', year=2019),
 Book(title='Echoes of Tomorrow', author='Seraphina Moon', year=2023),
 Book(title='Beneath the Obsidian Sky', author='Roric Stone', year=2017),
 Book(title="The Alchemist's Secret", author='Isabelle Dubois', year=2010),
 Book(title='Gardens of the Sunken City', author='Leo Maritime', year=2022),
 Book(title='A Flicker in the Dark', author='Nadia Volkov', year=2016),
 Book(title="The Cartographer's Dream", author='Elias Croft', year=2020)]

In [21]:
library.books[2]

Book(title='Chronicles of Aethel', author='Lyra Sterling', year=2015)

In [22]:
library.books[2].title, library.books[2].year

('Chronicles of Aethel', 2015)

In [25]:
titles = [book.title for book in library.books] # list comprehension
titles

['The Silent Forest',
 'Whispers of the Cosmos',
 'Chronicles of Aethel',
 'The Gilded Compass',
 'Echoes of Tomorrow',
 'Beneath the Obsidian Sky',
 "The Alchemist's Secret",
 'Gardens of the Sunken City',
 'A Flicker in the Dark',
 "The Cartographer's Dream"]

In [27]:
# filter books
newer_books = [(book.title, book.year) for book in library.books if book.year > 2018]
newer_books

[('Whispers of the Cosmos', 2021),
 ('The Gilded Compass', 2019),
 ('Echoes of Tomorrow', 2023),
 ('Gardens of the Sunken City', 2022),
 ("The Cartographer's Dream", 2020)]

In [29]:
library.model_dump() # get the json format dict back

{'name': 'Coolu Libraru',
 'books': [{'title': 'The Silent Forest',
   'author': 'Elara Vance',
   'year': 2018},
  {'title': 'Whispers of the Cosmos',
   'author': 'Dr. Kaelen Thorne',
   'year': 2021},
  {'title': 'Chronicles of Aethel', 'author': 'Lyra Sterling', 'year': 2015},
  {'title': 'The Gilded Compass', 'author': 'Arthur Pendelton', 'year': 2019},
  {'title': 'Echoes of Tomorrow', 'author': 'Seraphina Moon', 'year': 2023},
  {'title': 'Beneath the Obsidian Sky', 'author': 'Roric Stone', 'year': 2017},
  {'title': "The Alchemist's Secret",
   'author': 'Isabelle Dubois',
   'year': 2010},
  {'title': 'Gardens of the Sunken City',
   'author': 'Leo Maritime',
   'year': 2022},
  {'title': 'A Flicker in the Dark', 'author': 'Nadia Volkov', 'year': 2016},
  {'title': "The Cartographer's Dream",
   'author': 'Elias Croft',
   'year': 2020}]}

In [33]:
library.model_dump_json(indent=4)

'{\n    "name": "Coolu Libraru",\n    "books": [\n        {\n            "title": "The Silent Forest",\n            "author": "Elara Vance",\n            "year": 2018\n        },\n        {\n            "title": "Whispers of the Cosmos",\n            "author": "Dr. Kaelen Thorne",\n            "year": 2021\n        },\n        {\n            "title": "Chronicles of Aethel",\n            "author": "Lyra Sterling",\n            "year": 2015\n        },\n        {\n            "title": "The Gilded Compass",\n            "author": "Arthur Pendelton",\n            "year": 2019\n        },\n        {\n            "title": "Echoes of Tomorrow",\n            "author": "Seraphina Moon",\n            "year": 2023\n        },\n        {\n            "title": "Beneath the Obsidian Sky",\n            "author": "Roric Stone",\n            "year": 2017\n        },\n        {\n            "title": "The Alchemist\'s Secret",\n            "author": "Isabelle Dubois",\n            "year": 2010\n        }

In [35]:
with open ("library.json", "w") as json_file:
    json_file.write(library.model_dump_json(indent=4))

## Create pandas dataframe

In [41]:
import pandas as pd

titles = [book.title for book in library.books]
authors = [book.author for book in library.books]
years = [book.year for book in library.books]

pd.DataFrame(print[titles, authors, years])

TypeError: 'builtin_function_or_method' object is not subscriptable